# SVM Linear and Non-linear Experiments

This notebook runs **linear** and **non-linear SVM** models on the project dataset.

Workflow:
1. Load the dataset.
2. Define the original **4-class target** and the aggregated **3-class target** where classes `2` and `3` are merged.
3. Run SVM models **without imbalance handling**.
4. Run SVM models **with imbalance handling**, using combinations of oversampling and undersampling.
5. Evaluate every selected model with your `evaluate_classifier` function.
6. Compare the models by **test balanced accuracy**.

Hyperparameter tuning uses `GridSearchCV(scoring="balanced_accuracy")`, so each experiment selects the configuration maximizing cross-validated balanced accuracy.

In [ ]:
# ============================================================
# Imports
# ============================================================

from evaluate_classifier import evaluate_classifier

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC, SVC

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE, ADASYN, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler, TomekLinks, EditedNearestNeighbours
from imblearn.combine import SMOTETomek, SMOTEENN

import warnings
warnings.filterwarnings("ignore")


## 1. Load dataset

In [ ]:
DATA_PATH = "../dataset/cmi_internet_cleaned.csv"
TARGET_COL = "sii"
random_state = 42

dataset = pd.read_csv(DATA_PATH)

X = dataset.drop(TARGET_COL, axis=1)
y_4class = dataset[TARGET_COL]
y_3class = y_4class.replace({3: 2})

print("Features shape:", X.shape)
print("4-class target distribution:")
print(y_4class.value_counts().sort_index())
print("
4-class target proportions:")
print(y_4class.value_counts(normalize=True).sort_index())
print("
3-class target distribution:")
print(y_3class.value_counts().sort_index())
print("
3-class target proportions:")
print(y_3class.value_counts(normalize=True).sort_index())


## 2. Cross-validation setup

In [ ]:
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=random_state
)


## 3. Core experiment function

Resampling is applied **inside cross-validation folds** through an `imblearn` pipeline. This avoids leakage.

In [ ]:
def _make_pipeline_and_grid(model_type, imbalance_strategy):
    """Build pipeline and parameter grid for one SVM experiment."""

    if imbalance_strategy is None:
        if model_type == "linear":
            pipe = Pipeline([
                ("scaler", StandardScaler()),
                ("classifier", LinearSVC(random_state=random_state, max_iter=30000))
            ])
            grid = {
                "classifier__C": [0.001, 0.01, 0.1, 1, 10, 100],
                "classifier__class_weight": [None]
            }
        elif model_type == "nonlinear":
            pipe = Pipeline([
                ("scaler", StandardScaler()),
                ("classifier", SVC(random_state=random_state, probability=False))
            ])
            grid = [
                {
                    "classifier__kernel": ["rbf"],
                    "classifier__C": [0.1, 1, 10, 100],
                    "classifier__gamma": ["scale", 0.001, 0.01, 0.1],
                    "classifier__class_weight": [None]
                },
                {
                    "classifier__kernel": ["poly"],
                    "classifier__C": [0.1, 1, 10],
                    "classifier__gamma": ["scale", 0.001, 0.01],
                    "classifier__degree": [2, 3],
                    "classifier__class_weight": [None]
                }
            ]
        else:
            raise ValueError("model_type must be 'linear' or 'nonlinear'.")
        return pipe, grid

    if imbalance_strategy == "class_weight":
        if model_type == "linear":
            pipe = Pipeline([
                ("scaler", StandardScaler()),
                ("classifier", LinearSVC(random_state=random_state, max_iter=30000))
            ])
            grid = {
                "classifier__C": [0.001, 0.01, 0.1, 1, 10, 100],
                "classifier__class_weight": ["balanced"]
            }
        elif model_type == "nonlinear":
            pipe = Pipeline([
                ("scaler", StandardScaler()),
                ("classifier", SVC(random_state=random_state, probability=False))
            ])
            grid = [
                {
                    "classifier__kernel": ["rbf"],
                    "classifier__C": [0.1, 1, 10, 100],
                    "classifier__gamma": ["scale", 0.001, 0.01, 0.1],
                    "classifier__class_weight": ["balanced"]
                },
                {
                    "classifier__kernel": ["poly"],
                    "classifier__C": [0.1, 1, 10],
                    "classifier__gamma": ["scale", 0.001, 0.01],
                    "classifier__degree": [2, 3],
                    "classifier__class_weight": ["balanced"]
                }
            ]
        else:
            raise ValueError("model_type must be 'linear' or 'nonlinear'.")
        return pipe, grid

    samplers = {
        "ros": [("ros", RandomOverSampler(random_state=random_state))],
        "rus": [("rus", RandomUnderSampler(random_state=random_state))],
        "smote": [("smote", SMOTE(random_state=random_state))],
        "adasyn": [("adasyn", ADASYN(random_state=random_state))],
        "smote_tomek": [("smote_tomek", SMOTETomek(random_state=random_state))],
        "smote_enn": [("smote_enn", SMOTEENN(random_state=random_state))],
        "smote_then_tomek": [("smote", SMOTE(random_state=random_state)), ("tomek", TomekLinks())],
        "smote_then_enn": [("smote", SMOTE(random_state=random_state)), ("enn", EditedNearestNeighbours())],
        "adasyn_then_tomek": [("adasyn", ADASYN(random_state=random_state)), ("tomek", TomekLinks())],
        "adasyn_then_enn": [("adasyn", ADASYN(random_state=random_state)), ("enn", EditedNearestNeighbours())]
    }
    if imbalance_strategy not in samplers:
        raise ValueError(f"Unknown imbalance_strategy: {imbalance_strategy}")

    if model_type == "linear":
        classifier = LinearSVC(random_state=random_state, max_iter=30000)
        classifier_grid = {
            "classifier__C": [0.001, 0.01, 0.1, 1, 10],
            "classifier__class_weight": [None]
        }
    elif model_type == "nonlinear":
        classifier = SVC(random_state=random_state, probability=False)
        classifier_grid = [
            {
                "classifier__kernel": ["rbf"],
                "classifier__C": [0.1, 1, 10, 100],
                "classifier__gamma": ["scale", 0.001, 0.01, 0.1],
                "classifier__class_weight": [None]
            },
            {
                "classifier__kernel": ["poly"],
                "classifier__C": [0.1, 1, 10],
                "classifier__gamma": ["scale", 0.001, 0.01],
                "classifier__degree": [2, 3],
                "classifier__class_weight": [None]
            }
        ]
    else:
        raise ValueError("model_type must be 'linear' or 'nonlinear'.")

    pipe = ImbPipeline([
        ("scaler", StandardScaler()),
        *samplers[imbalance_strategy],
        ("classifier", classifier)
    ])

    def add_sampler_params(g):
        g = g.copy()
        if "smote" in pipe.named_steps:
            g["smote__k_neighbors"] = [3, 5]
        if "adasyn" in pipe.named_steps:
            g["adasyn__n_neighbors"] = [3, 5]
        if "enn" in pipe.named_steps:
            g["enn__n_neighbors"] = [3, 5]
            g["enn__kind_sel"] = ["mode"]
        return g

    if isinstance(classifier_grid, dict):
        grid = add_sampler_params(classifier_grid)
    else:
        grid = [add_sampler_params(g) for g in classifier_grid]

    return pipe, grid


def run_svm_experiment(
    X, y, target_name, model_type, imbalance_strategy=None,
    test_size=0.30, cv=cv, n_jobs=-1, verbose=1, plot_confusion=True
):
    """Tune one SVM with GridSearchCV and evaluate the best estimator using evaluate_classifier."""

    experiment_name = f"SVM {model_type.upper()} - {target_name}"
    experiment_name += " - No Imbalance Handling" if imbalance_strategy is None else f" - {imbalance_strategy}"

    print("=" * 100)
    print(experiment_name)
    print("=" * 100)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )

    print("Train shape:", X_train.shape)
    print("Test shape:", X_test.shape)
    print("\nTrain distribution:")
    print(y_train.value_counts(normalize=True).sort_index())
    print("\nTest distribution:")
    print(y_test.value_counts(normalize=True).sort_index())

    pipe, param_grid = _make_pipeline_and_grid(model_type, imbalance_strategy)

    grid = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        scoring="balanced_accuracy",
        cv=cv,
        n_jobs=n_jobs,
        verbose=verbose,
        error_score="raise"
    )
    grid.fit(X_train, y_train)

    print("\nBest parameters:")
    print(grid.best_params_)
    print("\nBest CV balanced accuracy:")
    print(grid.best_score_)

    best_model = grid.best_estimator_

    test_results, report = evaluate_classifier(
        model=best_model,
        X_test=X_test,
        y_test=y_test,
        model_name=experiment_name,
        labels=np.sort(y.unique()),
        average="weighted",
        plot_confusion=plot_confusion,
        plot_roc=False,
        plot_pr=False,
        normalize_cm="true"
    )

    summary = {
        "model_name": experiment_name,
        "target": target_name,
        "model_type": model_type,
        "imbalance_strategy": "none" if imbalance_strategy is None else imbalance_strategy,
        "best_cv_balanced_accuracy": grid.best_score_,
        "best_params": grid.best_params_,
        **test_results
    }

    return {
        "name": experiment_name,
        "grid": grid,
        "best_model": best_model,
        "test_results": test_results,
        "report": report,
        "summary": summary,
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test
    }


## 4. Choose which experiments to run

In [ ]:
RUN_4_CLASS = True
RUN_3_CLASS = True

# Remove some strategies if the notebook takes too long.
imbalance_strategies = [
    "class_weight",
    "ros",
    "rus",
    "smote",
    "adasyn",
    "smote_tomek",
    "smote_enn",
    "smote_then_tomek",
    "smote_then_enn",
    "adasyn_then_tomek",
    "adasyn_then_enn"
]


## 5. SVM without imbalance handling

In [ ]:
all_experiments = []

if RUN_4_CLASS:
    linear_4_no_imb = run_svm_experiment(X, y_4class, "4 Classes", "linear", None, cv=cv)
    all_experiments.append(linear_4_no_imb)

    nonlinear_4_no_imb = run_svm_experiment(X, y_4class, "4 Classes", "nonlinear", None, cv=cv)
    all_experiments.append(nonlinear_4_no_imb)

if RUN_3_CLASS:
    linear_3_no_imb = run_svm_experiment(X, y_3class, "3 Classes", "linear", None, cv=cv)
    all_experiments.append(linear_3_no_imb)

    nonlinear_3_no_imb = run_svm_experiment(X, y_3class, "3 Classes", "nonlinear", None, cv=cv)
    all_experiments.append(nonlinear_3_no_imb)


## 6. SVM with imbalance handling

In [ ]:
for strategy in imbalance_strategies:
    if RUN_4_CLASS:
        exp = run_svm_experiment(X, y_4class, "4 Classes", "linear", strategy, cv=cv)
        all_experiments.append(exp)

        exp = run_svm_experiment(X, y_4class, "4 Classes", "nonlinear", strategy, cv=cv)
        all_experiments.append(exp)

    if RUN_3_CLASS:
        exp = run_svm_experiment(X, y_3class, "3 Classes", "linear", strategy, cv=cv)
        all_experiments.append(exp)

        exp = run_svm_experiment(X, y_3class, "3 Classes", "nonlinear", strategy, cv=cv)
        all_experiments.append(exp)


## 7. Final comparison

In [ ]:
svm_results_table = pd.DataFrame([exp["summary"] for exp in all_experiments])

print("Available columns:")
print(svm_results_table.columns.tolist())

sort_col = "balanced_accuracy" if "balanced_accuracy" in svm_results_table.columns else "best_cv_balanced_accuracy"

svm_results_table = svm_results_table.sort_values(by=sort_col, ascending=False).reset_index(drop=True)
svm_results_table


In [ ]:
best_svm_overall = svm_results_table.iloc[0]
best_svm_overall


## 8. Separate comparisons

In [ ]:
for target in svm_results_table["target"].unique():
    for model_type in svm_results_table["model_type"].unique():
        subset = svm_results_table[
            (svm_results_table["target"] == target) &
            (svm_results_table["model_type"] == model_type)
        ].sort_values(by=sort_col, ascending=False)

        print("=" * 90)
        print(f"Best {model_type} SVM - {target}")
        print("=" * 90)
        display(subset.head(5))


## 9. Save results

In [ ]:
OUTPUT_RESULTS_PATH = "svm_linear_nonlinear_results.csv"
svm_results_table.to_csv(OUTPUT_RESULTS_PATH, index=False)
print(f"Saved results to: {OUTPUT_RESULTS_PATH}")
